# Pertemuan 12 - Asosiasi Data dan Sistem Rekomendasi Dasar

**Nama:** Nabil Fakhrezy  
**NIM:** 240401010286  
**Kelas:** IF401  
**Program Studi:** PJJ Informatika

## Materi
Apriori, Market Basket Analysis, dan Content-Based Filtering.


## 1. Import Library dan Generate Dataset Transaksi


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from itertools import combinations

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

sns.set_theme(style="whitegrid")
np.random.seed(42)

class TransactionEncoder:
    def fit(self, transaksi):
        self.columns_ = sorted(set(item for trx in transaksi for item in trx))
        return self
    def transform(self, transaksi):
        return np.array([[item in trx for item in self.columns_] for trx in transaksi])
    def fit_transform(self, transaksi):
        return self.fit(transaksi).transform(transaksi)

def apriori(df_bool, min_support=0.1, use_colnames=True):
    cols = list(df_bool.columns)
    n = len(df_bool)
    rows = []
    for r in range(1, len(cols) + 1):
        found = 0
        for combo in combinations(cols, r):
            support = df_bool[list(combo)].all(axis=1).sum() / n
            if support >= min_support:
                found += 1
                rows.append({"support": support, "itemsets": frozenset(combo)})
        if found == 0 and r > 2:
            break
    return pd.DataFrame(rows)

def association_rules(freq_items, metric="confidence", min_threshold=0.5):
    support_map = {row["itemsets"]: row["support"] for _, row in freq_items.iterrows()}
    rows = []
    for itemset, support in support_map.items():
        if len(itemset) < 2:
            continue
        items = list(itemset)
        for r in range(1, len(items)):
            for ant_tuple in combinations(items, r):
                antecedent = frozenset(ant_tuple)
                consequent = itemset - antecedent
                ant_sup = support_map.get(antecedent, 0)
                con_sup = support_map.get(consequent, 0)
                if ant_sup == 0 or con_sup == 0:
                    continue
                confidence = support / ant_sup
                lift = confidence / con_sup if con_sup else np.nan
                if confidence >= min_threshold:
                    rows.append({
                        "antecedents": antecedent,
                        "consequents": consequent,
                        "antecedent support": ant_sup,
                        "consequent support": con_sup,
                        "support": support,
                        "confidence": confidence,
                        "lift": lift
                    })
    return pd.DataFrame(rows)

produk = ["Roti", "Selai", "Susu", "Sereal", "Telur", "Keju", "Kopi", "Gula", "Teh", "Mentega"]
transaksi = []

for i in range(50):
    item = list(np.random.choice(produk, np.random.randint(2, 6), replace=False))
    transaksi.append(item)

for i in range(20):
    if "Roti" not in transaksi[i]:
        transaksi[i].append("Roti")
    if "Selai" not in transaksi[i]:
        transaksi[i].append("Selai")

for i in range(20, 35):
    if "Kopi" not in transaksi[i]:
        transaksi[i].append("Kopi")
    if "Gula" not in transaksi[i]:
        transaksi[i].append("Gula")

df_transaksi = pd.DataFrame({"transaksi_id": [f"T{i+1:03d}" for i in range(50)], "daftar_item": transaksi})
display(df_transaksi.head(10))

,transaksi_id,daftar_item
0,T001,"[Keju, Roti, Mentega, Kopi, Selai]"
1,T002,"[Roti, Kopi, Teh, Selai, Mentega]"
2,T003,"[Kopi, Susu, Teh, Roti, Selai]"
3,T004,"[Selai, Keju, Telur, Teh, Roti]"
4,T005,"[Mentega, Susu, Gula, Keju, Roti, Selai]"
5,T006,"[Mentega, Gula, Roti, Selai]"
6,T007,"[Teh, Mentega, Keju, Telur, Roti, Selai]"
7,T008,"[Roti, Susu, Gula, Selai]"
8,T009,"[Susu, Teh, Telur, Roti, Selai]"
9,T010,"[Kopi, Susu, Roti, Selai]"


## 2. Apriori dan Association Rules


In [2]:
counter_produk = Counter()
for items in transaksi:
    counter_produk.update(items)

freq_produk = pd.DataFrame(counter_produk.items(), columns=["produk", "frekuensi"]).sort_values("frekuensi", ascending=False)
display(freq_produk)

te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df_onehot = pd.DataFrame(te_ary, columns=te.columns_)

freq_items = apriori(df_onehot, min_support=0.10, use_colnames=True)
freq_items = freq_items.sort_values("support", ascending=False)

rules = association_rules(freq_items, metric="confidence", min_threshold=0.50)
rules = rules[rules["lift"] > 1].copy()
rules = rules.sort_values(["lift", "confidence"], ascending=False)

rules["antecedents_str"] = rules["antecedents"].apply(lambda x: ", ".join(list(x)))
rules["consequents_str"] = rules["consequents"].apply(lambda x: ", ".join(list(x)))

display(freq_items.head(10))
display(rules[["antecedents_str", "consequents_str", "support", "confidence", "lift"]].head(10).round(3))


,produk,frekuensi
4,Selai,33
1,Roti,29
8,Gula,27
5,Teh,23
3,Kopi,23
2,Mentega,21
7,Telur,18
0,Keju,17
6,Susu,16
9,Sereal,9


,support,itemsets
5,0.66,(Selai)
4,0.58,(Roti)
0,0.54,(Gula)
38,0.48,"(Roti, Selai)"
8,0.46,(Teh)
2,0.46,(Kopi)
3,0.42,(Mentega)
9,0.36,(Telur)
1,0.34,(Keju)
11,0.34,"(Gula, Kopi)"


,antecedents_str,consequents_str,support,confidence,lift
59,"Keju, Teh",Telur,0.12,0.857,2.381
80,"Telur, Gula, Teh",Kopi,0.10,1.000,2.174
85,"Roti, Teh","Telur, Selai",0.10,0.500,2.083
77,"Telur, Kopi","Gula, Teh",0.10,0.556,1.984
55,Sereal,Mentega,0.14,0.778,1.852
34,"Telur, Kopi",Gula,0.18,1.000,1.852
81,"Telur, Kopi, Teh",Gula,0.10,1.000,1.852
33,"Telur, Gula",Kopi,0.18,0.818,1.779
58,"Telur, Teh",Keju,0.12,0.600,1.765
41,"Susu, Selai",Roti,0.18,1.000,1.724


## 3. Content-Based Filtering


In [3]:
katalog = pd.DataFrame({
    "produk": produk,
    "kategori": ["Bakery", "Bakery", "Dairy", "Bakery", "Dairy", "Dairy", "Minuman", "Bumbu", "Minuman", "Dairy"],
    "harga": [15000, 18000, 12000, 25000, 24000, 30000, 22000, 14000, 16000, 28000]
})

fitur_kategori = pd.get_dummies(katalog["kategori"], prefix="kategori")
scaler = MinMaxScaler()
fitur_harga = pd.DataFrame(scaler.fit_transform(katalog[["harga"]]), columns=["harga_scaled"])
fitur_produk = pd.concat([fitur_kategori, fitur_harga], axis=1)

sim_matrix = cosine_similarity(fitur_produk)

def rekomendasi_serupa(nama_produk, top_n=3):
    idx = katalog.index[katalog["produk"] == nama_produk][0]
    skor = list(enumerate(sim_matrix[idx]))
    skor = sorted(skor, key=lambda x: x[1], reverse=True)
    skor = [s for s in skor if s[0] != idx][:top_n]
    hasil = katalog.iloc[[i for i, _ in skor]].copy()
    hasil["similarity_score"] = [score for _, score in skor]
    return hasil

display(katalog)
display(rekomendasi_serupa("Roti", top_n=3).round(3))


,produk,kategori,harga
0,Roti,Bakery,15000
1,Selai,Bakery,18000
2,Susu,Dairy,12000
3,Sereal,Bakery,25000
4,Telur,Dairy,24000
5,Keju,Dairy,30000
6,Kopi,Minuman,22000
7,Gula,Bumbu,14000
8,Teh,Minuman,16000
9,Mentega,Dairy,28000


,produk,kategori,harga,similarity_score
1,Selai,Bakery,18000,0.988
3,Sereal,Bakery,25000,0.896
5,Keju,Dairy,30000,0.116


## 4. Perbandingan Rekomendasi


In [4]:
produk_target = "Roti"
rules_terkait = rules[rules["antecedents"].apply(lambda x: produk_target in x)].copy()
display(rules_terkait[["consequents_str", "support", "confidence", "lift"]].head(5).round(3))

print("Rekomendasi Content-Based:")
display(rekomendasi_serupa(produk_target, top_n=3).round(3))


,consequents_str,support,confidence,lift
85,"Telur, Selai",0.1,0.500,2.083
83,"Selai, Teh",0.1,0.500,1.667
89,Telur,0.1,0.556,1.543
23,Selai,0.2,1.000,1.515
87,Selai,0.1,1.000,1.515


Rekomendasi Content-Based:


,produk,kategori,harga,similarity_score
1,Selai,Bakery,18000,0.988
3,Sereal,Bakery,25000,0.896
5,Keju,Dairy,30000,0.116


## Kesimpulan

Saya mempelajari association rule mining dan rekomendasi sederhana. Temuan utama adalah Apriori menemukan pola produk yang sering dibeli bersama, sedangkan Content-Based Filtering memberi rekomendasi berdasarkan kemiripan atribut produk. Keterbatasannya, dataset transaksi masih sintetis.
